# Planner Agent

LangGraph-based agent that breaks a travel request into actionable tasks (airfare, hotel, car rental).

In [1]:
import asyncio
import logging
import os
from collections.abc import AsyncIterable
from typing import Any, Literal

import uvicorn
import weave
from dotenv import load_dotenv
from langchain_core.messages import AIMessage
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent
from pydantic import BaseModel, Field

import prompts
from common import BaseAgent, TaskList, build_a2a_app

load_dotenv()
logger = logging.getLogger(__name__)

WANDB_PROJECT = os.getenv('WANDB_WORKSPACE')
weave_client = weave.init(WANDB_PROJECT)
print(f'Weave initialized: {WANDB_PROJECT}')

/Users/paul/.cache/uv/archive-v0/DwhQhR0PIXGhL5z9-p-BG/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
weave: weave version 0.52.37 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: paulbruffett.
weave: View Weave data at https://wandb.ai/pbruffett/a2a-lab/weave


Weave initialized: pbruffett/a2a-lab


In [2]:
class ResponseFormat(BaseModel):
    """Respond to the user in this format."""
    status: Literal['input_required', 'completed'] = 'input_required'
    question: str = Field(default='', description='Input needed from the user')
    content: TaskList = Field(default_factory=TaskList, description='List of tasks when the plan is generated')


class LangGraphPlannerAgent(BaseAgent):
    def __init__(self):
        super().__init__(agent_name='PlannerAgent', description='Breakdown user request into tasks', content_types=['text', 'text/plain'])
        self.model = ChatAnthropic(model='claude-sonnet-4-6', temperature=0.0)
        self.graph = create_agent(
            self.model, checkpointer=MemorySaver(),
            system_prompt=prompts.PLANNER_COT_INSTRUCTIONS,
            response_format=ResponseFormat, tools=[],
        )

    @weave.op()
    async def stream(self, query, session_id, task_id) -> AsyncIterable[dict[str, Any]]:
        logger.info('[planner.stream] REQUEST session=%s task=%s query=%r', session_id, task_id, query)
        config = {'configurable': {'thread_id': session_id}}
        async for _ in self.graph.astream({'messages': [('user', query)]}, config, stream_mode='values'):
            pass  # consume the graph to completion
        resp = self.graph.get_state(config).values.get('structured_response')
        if isinstance(resp, ResponseFormat) and resp.status == 'completed':
            logger.info('[planner.stream] COMPLETE')
            yield {'response_type': 'data', 'is_task_complete': True, 'require_user_input': False, 'content': resp.content.model_dump()}
        else:
            question = resp.question if isinstance(resp, ResponseFormat) else 'Unable to process request. Please try again.'
            logger.info('[planner.stream] INPUT_REQUIRED question=%r', question) 
            yield {'response_type': 'text', 'is_task_complete': False, 'require_user_input': True, 'content': question}

In [ ]:
app = build_a2a_app(LangGraphPlannerAgent(), 'agent_cards/planner_agent.json')

if __name__ == "__main__":
    config = uvicorn.Config(app=app, host='localhost', port=10102)
    server = uvicorn.Server(config)
    await server.serve()

INFO:     Started server process [35906]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:10102 (Press CTRL+C to quit)
weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019db081-79be-7fc0-9d4c-4df184361bbe
Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]


INFO:     ::1:54567 - "POST / HTTP/1.1" 200 OK


Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]
weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019db082-3547-7a64-94a9-3594950fa420
Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]


INFO:     ::1:54647 - "POST / HTTP/1.1" 200 OK
